<a href="https://colab.research.google.com/github/arjunnamburi/ml_death_overs_ipl/blob/main/notebooks/features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
df = pd.read_csv(DATA + 'prog3.csv')

In [48]:
df['venue'].nunique()

60

##Fixing Venue Names


In [49]:
import re
import pandas as pd

# 1) Grouping key: same key = same physical venue
ALIASES = [
    (r"feroz shah kotla|arun jaitley",                  "delhi_arun_jaitley"),
    (r"narendra modi|sardar patel|motera",              "ahmedabad_narendra_modi"),
    (r"sheikh zayed|zayed cricket",                     "abu_dhabi_zayed"),
    (r"yadavindra singh",                               "mullanpur_new_chandigarh"),
    (r"bindra|punjab cricket association",              "mohali_pca"),
    (r"chinnaswamy",                                    "bengaluru_chinnaswamy"),
    (r"chidambaram|chepauk",                            "chennai_chidambaram"),
    (r"rajiv gandhi",                                   "hyderabad_uppal"),
    (r"eden gardens",                                   "kolkata_eden"),
    (r"wankhede",                                       "mumbai_wankhede"),
    (r"brabourne",                                      "mumbai_brabourne"),
    (r"dy patil|d\.y\. patil",                          "navi_mumbai_dy_patil"),
    (r"maharashtra cricket association",                "pune_mca"),
    (r"sawai mansingh",                                 "jaipur_sms"),
    (r"himachal pradesh",                               "dharamsala_hpca"),
    (r"rajasekhara reddy|aca-vdca",                     "visakhapatnam_aca_vdca"),
    (r"shaheed veer narayan",                           "raipur_svns"),
    (r"vajpayee|ekana",                                 "lucknow_ekana"),
]

def venue_key(name: str) -> str:
    n = str(name).lower().strip()
    for pattern, key in ALIASES:
        if re.search(pattern, n):
            return key
    # fallback: drop ", City" suffix, remove punctuation
    n = n.split(",")[0]
    return re.sub(r"[^a-z0-9 ]", "", n).strip()

df["venue_key"] = df["venue"].map(venue_key)

# 2) Most frequent original name within each key
canonical = (
    df.groupby("venue_key")["venue"]
      .agg(lambda s: s.value_counts().idxmax())
)

# 3) Replace
df["venue"] = df["venue_key"].map(canonical)
df = df.drop(columns="venue_key")



In [50]:
df['venue'].nunique()

37

## SUBGROUP 1

In [51]:
import numpy as np
import pandas as pd

In [52]:
def shifted_cumsum(df, keys, col):
    """Cumulative total of col within each group, excluding the current row."""
    return df.groupby(keys)[col].transform(lambda s: s.cumsum().shift(1, fill_value=0))

df['_is_wkt'] = df['player_out'].notna().astype(int)   # adjust if retired hurt is recorded here

team, bat, bowl = ['match_id', 'innings'], ['match_id', 'innings', 'batter'], ['match_id', 'bowler']
rebuilt = {
    'c_team_runs':      shifted_cumsum(df, team, 'ball_runs_total'),
    'c_team_balls':     shifted_cumsum(df, team, 'valid_ball'),
    'c_team_wickets':   shifted_cumsum(df, team, '_is_wkt'),
    'c_batter_runs':    shifted_cumsum(df, bat,  'ball_runs_batter'),
    'c_batter_balls':   shifted_cumsum(df, bat,  'binary_ball_faced'),
    'c_bowler_runs':    shifted_cumsum(df, bowl, 'ball_runs_bowler'),
    'c_bowler_wickets': shifted_cumsum(df, bowl, 'binary_bowler_wicket'),
    'c_bowler_balls':   shifted_cumsum(df, bowl, 'valid_ball'),
}

for col, new in rebuilt.items():
    bad = df[col] != new
    print(f"{col:18s} {bad.sum():6d} rows wrong in {df.loc[bad, 'match_id'].nunique():4d} matches")

c_team_runs            22 rows wrong in    1 matches
c_team_balls           77 rows wrong in    4 matches
c_team_wickets       1147 rows wrong in   19 matches
c_batter_runs          14 rows wrong in    1 matches
c_batter_balls         45 rows wrong in    4 matches
c_bowler_runs           0 rows wrong in    0 matches
c_bowler_wickets        0 rows wrong in    0 matches
c_bowler_balls          0 rows wrong in    0 matches


In [53]:
def first_divergences(col):
    d = df[df[col] != rebuilt[col]].copy()
    d['stored'], d['rebuilt'] = d[col], rebuilt[col][d.index]
    first = d.groupby(['match_id', 'innings']).head(1)
    return first[['match_id', 'innings', 'over', 'ball', 'batter', 'player_out',
                  'valid_ball', 'stored', 'rebuilt']]

# Problem 1: are these the same matches, and what happens at the first wrong row?
small = ['c_team_runs', 'c_team_balls', 'c_batter_runs', 'c_batter_balls']
for col in small:
    print(col, sorted(df.loc[df[col] != rebuilt[col], 'match_id'].unique()))
print(first_divergences('c_team_balls'))

# Problem 2: which direction, and what wicket caused it?
fd = first_divergences('c_team_wickets')
print((fd.stored - fd.rebuilt).value_counts())    # +1 = stored counts more, -1 = rebuilt counts more

# The dismissal just before each first divergence, and whether that player bats again later
for _, r in fd.iterrows():
    inn = df[(df.match_id == r.match_id) & (df.innings == r.innings)]
    prev = inn.loc[:r.name].iloc[:-1]
    wk = prev[prev.player_out.notna()].iloc[-1]
    later = inn.loc[wk.name + 1:]
    returns = ((later.batter == wk.player_out) | (later.non_striker == wk.player_out)).any()
    print(r.match_id, r.innings, 'over', wk.over, 'out:', wk.player_out, '| bats again:', returns)

c_team_runs [np.int64(1136564)]
c_team_balls [np.int64(335994), np.int64(392198), np.int64(419155), np.int64(1136564)]
c_batter_runs [np.int64(1136564)]
c_batter_balls [np.int64(335994), np.int64(392198), np.int64(419155), np.int64(1136564)]
        match_id  innings  over  ball        batter player_out  valid_ball  \
3035      335994        2    11     2  AC Gilchrist        NaN           1   
17070     392198        2    11     2     YK Pathan        NaN           1   
39063     419155        1    19     2  DE Bollinger        NaN           1   
151306   1136564        2    12     2      S Dhawan        NaN           1   

        stored  rebuilt  
3035      68.0       67  
17070     68.0       67  
39063    116.0      115  
151306    74.0       73  
-1.0    19
Name: count, dtype: int64
335985 1 over 3 out: DJ Thornely | bats again: False
419107 1 over 19 out: Harbhajan Singh | bats again: False
419117 1 over 3 out: VVS Laxman | bats again: False
548306 2 over 8 out: SR Tendulkar | b

In [54]:
for col in ['c_team_runs', 'c_team_balls', 'c_batter_runs', 'c_batter_balls']:
    df[col] = rebuilt[col].astype(int)

In [55]:

retired_rows = []
for _, r in fd.iterrows():
    inn = df[(df.match_id == r.match_id) & (df.innings == r.innings)]
    prev = inn.loc[:r.name].iloc[:-1]
    retired_rows.append(prev[prev.player_out.notna()].index[-1])

df['is_retired'] = df.index.isin(retired_rows).astype(int)
assert df['is_retired'].sum() == 19
print(df.loc[df.is_retired == 1, ['match_id', 'innings', 'over', 'ball', 'player_out']])

        match_id  innings  over  ball       player_out
714       335985        1     3     4      DJ Thornely
27463     419107        1    19     5  Harbhajan Singh
29747     419117        1     3     3       VVS Laxman
58764     548306        2     8     6     SR Tendulkar
91424     598060        2    11     4     SR Tendulkar
98156     729309        1    19     5  Harbhajan Singh
110775    829721        1     3     3         AJ Finch
126046    980931        2     1     5     KP Pietersen
145226   1082628        2    13     3        KH Pandya
154022   1136576        2     0     5         S Dhawan
157248   1136589        2    16     2           N Rana
204472   1254104        1     1     6        AT Rayudu
212931   1304067        2    13     1      RA Tripathi
227548   1359482        1    10     1    PBB Rajapaksa
240814   1359537        1    15     6        KH Pandya
268304   1473469        2     5     3        SV Samson
274939   1473499        2    17     5      PHKD Mendis
282450   1

In [56]:
df['is_dismissal'] = (df['player_out'].notna() & (df['is_retired'] == 0)).astype(int)

In [57]:
df['c_team_wickets'] = df['c_team_wickets'].astype(int)

In [58]:
ret = df['is_retired'] == 1

df.loc[ret, 'striker_out'] = 0

# Match target's existing type: ints if classes are 0,1,2..., strings if '0','1',...
df.loc[ret, 'target'] = df.loc[ret, 'ball_runs_batter']            # if target holds ints
# df.loc[ret, 'target'] = df.loc[ret, 'ball_runs_batter'].astype(str)   # if it holds strings

print(df.loc[ret, ['player_out', 'ball_runs_batter', 'striker_out', 'target']])
print(df['target'].unique())   # should show no new or mixed types

             player_out  ball_runs_batter  striker_out target
714         DJ Thornely                 0            0      0
27463   Harbhajan Singh                 0            0      0
29747        VVS Laxman                 1            0      1
58764      SR Tendulkar                 1            0      1
91424      SR Tendulkar                 6            0      6
98156   Harbhajan Singh                 0            0      0
110775         AJ Finch                 1            0      1
126046     KP Pietersen                 0            0      0
145226        KH Pandya                 1            0      1
154022         S Dhawan                 0            0      0
157248           N Rana                 4            0      4
204472        AT Rayudu                 0            0      0
212931      RA Tripathi                 6            0      6
227548    PBB Rajapaksa                 0            0      0
240814        KH Pandya                 0            0      0
268304  

In [59]:
# ---- Config (safe to rerun) ----
CONFIG = globals().get('CONFIG', {})
CONFIG.update({
    'FULL_BALLS': 120,
    'PHASE_BINS': [0, 6, 15, 20],                 # overs 0-5, 6-14, 15-19
    'PHASE_LABELS': ['powerplay', 'middle', 'death'],
})
CONFIG.setdefault('BAT_FORM_BALLS', 100)
CONFIG.setdefault('BOWL_FORM_BALLS', 120)
for k in ['K_BATTER', 'K_BOWLER', 'K_VENUE', 'K_SEASON']:
    CONFIG.setdefault(k, 100)

# Detect whether 'over' starts at 0 or 1
CONFIG['OVER_OFFSET'] = int(df['over'].min())
assert CONFIG['OVER_OFFSET'] in (0, 1), f"unexpected first over: {df['over'].min()}"
print('OVER_OFFSET =', CONFIG['OVER_OFFSET'], '| over range:', df['over'].min(), '-', df['over'].max())


# ---- Subgroup 1 (revised version) ----
def add_match_state_features(df, cfg=CONFIG):
    """Subgroup 1. Uses only state entering the ball, so no leakage."""
    df = df.copy()
    balls = df['c_team_balls']
    runs = df['c_team_runs']

    df['balls_remaining'] = (cfg['FULL_BALLS'] - balls).clip(lower=1)
    df['wickets_remaining'] = 10 - df['c_team_wickets']
    df['current_run_rate'] = runs * 6 / balls.where(balls > 0)        # NaN on first ball

    current_over = (df['over'] - cfg['OVER_OFFSET']).clip(upper=19)
    df['match_phase'] = pd.cut(current_over, bins=cfg['PHASE_BINS'], right=False,
                               labels=cfg['PHASE_LABELS'])

    legal_before = (df.groupby(['match_id', 'innings', 'over'])['valid_ball']
                      .transform(lambda s: s.cumsum().shift(1, fill_value=0)))
    df['ball_in_over'] = (legal_before + 1).clip(upper=6)

    df['projected_score'] = runs + df['current_run_rate'] * df['balls_remaining'] / 6
    df['balls_per_wicket_remaining'] = df['balls_remaining'] / df['wickets_remaining']
    return df


df = df.sort_values(['match_id', 'innings', 'over', 'ball'], kind='stable').reset_index(drop=True)
df = add_match_state_features(df)


# ---- Checks ----
OFF = CONFIG['OVER_OFFSET']
g = df.groupby(['match_id', 'innings'])

# 1. Ranges                                   Expected: True True True True
print('1.', df['balls_remaining'].between(1, 120).all(),
      df['wickets_remaining'].between(1, 10).all(),
      df['ball_in_over'].between(1, 6).all(),
      df['match_phase'].notna().all())

# 2. Legal-ball count agrees with over number  Expected: 0
print('2.', ((df['c_team_balls'] // 6) != (df['over'] - OFF)).sum())

# 3. ball_in_over agrees with legal-ball count Expected: 0
print('3.', (df['ball_in_over'] != df['c_team_balls'] % 6 + 1).sum())

# 4. Run rate NaN only before first legal ball Expected: [0] True
print('4.', df.loc[df['current_run_rate'].isna(), 'c_team_balls'].unique(),
      df['current_run_rate'].isna().sum() >= g.ngroups)

# 5. First delivery of every innings          Expected: True True True True
first = g.head(1)
print('5.', (first['balls_remaining'] == 120).all(),
      (first['wickets_remaining'] == 10).all(),
      (first['match_phase'] == 'powerplay').all(),
      (first['ball_in_over'] == 1).all())

# 6. Phase boundaries                          Expected: 0-5 powerplay, 6-14 middle, 15-19 death only
print('6.'); print(pd.crosstab(df['over'] - OFF, df['match_phase']))

# 7. State only moves one way                  Expected: 0 0
print('7.', (g['balls_remaining'].diff() > 0).sum(),
      (g['wickets_remaining'].diff() > 0).sum())

# 8. Derived features                          Expected: True True; min >= 0.1, max <= 120
has_ps = df['projected_score'].notna()
print('8.', (df['projected_score'].isna() == df['current_run_rate'].isna()).all(),
      (df.loc[has_ps, 'projected_score'] >= df.loc[has_ps, 'c_team_runs']).all())
print(df['balls_per_wicket_remaining'].agg(['min', 'max']))

# 9. Dtypes                                    Expected: match_phase category, rest int/float
print('9.'); print(df[['balls_remaining', 'wickets_remaining', 'current_run_rate', 'match_phase',
                       'ball_in_over', 'projected_score', 'balls_per_wicket_remaining']].dtypes)

# 10. Rain-affected rows computed too          Expected: 0
print('10.', df.loc[df['rain_affected'] == 1, ['balls_remaining', 'match_phase']].isna().sum().sum())

# 11. Hand-check: an over with a wide + start of innings 2
w = df[(df['valid_ball'] == 0) & (df['innings'] == 1)].iloc[0]
cols = ['over', 'ball', 'valid_ball', 'c_team_runs', 'c_team_balls', 'c_team_wickets',
        'balls_remaining', 'wickets_remaining', 'current_run_rate', 'match_phase',
        'ball_in_over', 'projected_score']
inn1 = df[(df['match_id'] == w['match_id']) & (df['innings'] == 1)]
print('11.'); print(inn1[inn1['over'] == w['over']][cols])
print(df[(df['match_id'] == w['match_id']) & (df['innings'] == 2)].head(3)[cols])

OVER_OFFSET = 0 | over range: 0 - 19
1. True True True True
2. 35
3. 202
4. [0] True
5. True True True True
6.
match_phase  powerplay  middle  death
over                                 
0                15613       0      0
1                15614       0      0
2                15416       0      0
3                15396       0      0
4                15382       0      0
5                15308       0      0
6                    0   15251      0
7                    0   15228      0
8                    0   15190      0
9                    0   15155      0
10                   0   15082      0
11                   0   15041      0
12                   0   15024      0
13                   0   14894      0
14                   0   14787      0
15                   0       0  14604
16                   0       0  14348
17                   0       0  13918
18                   0       0  13064
19                   0       0  11238
7. 0 0
8. True True
min     0.1
max    65.0
Name: bal

In [60]:
po = df.groupby(['match_id', 'innings', 'over'])['valid_ball'].sum().reset_index(name='legal')
last_over = po.groupby(['match_id', 'innings'])['over'].transform('max')
short = po[(po['legal'] < 6) & (po['over'] != last_over)]   # short overs that aren't the innings' final over
print(len(short), 'short overs in', short[['match_id', 'innings']].drop_duplicates().shape[0], 'innings')
print(short)

# Are all mismatches inside those innings, after the short over?
mm = df[(df['c_team_balls'] // 6) != df['over']]
key = mm.merge(short[['match_id', 'innings']].drop_duplicates(), on=['match_id', 'innings'], how='left', indicator=True)
print(key['_merge'].value_counts())   # expect all 'both'

4 short overs in 4 innings
      match_id  innings  over  legal
197     335987        1     7      5
1285    336015        2    14      5
6872    501202        1     5      5
8886    501255        2     9      5
_merge
both          35
left_only      0
right_only     0
Name: count, dtype: int64


In [61]:
def add_match_state_features(df, cfg=CONFIG):
    """Subgroup 1. Uses only state entering the ball, so no leakage."""
    df = df.copy()
    balls = df['c_team_balls']
    runs = df['c_team_runs']

    current_over = (df['over'] - cfg['OVER_OFFSET']).clip(upper=19)
    df['match_phase'] = pd.cut(current_over, bins=cfg['PHASE_BINS'], right=False,
                               labels=cfg['PHASE_LABELS'])

    legal_before = (df.groupby(['match_id', 'innings', 'over'])['valid_ball']
                      .transform(lambda s: s.cumsum().shift(1, fill_value=0)))
    df['ball_in_over'] = (legal_before + 1).clip(upper=6)

    # From the scorers' over and ball position, robust to short overs
    df['balls_remaining'] = (cfg['FULL_BALLS']
                             - (current_over * 6 + df['ball_in_over'] - 1)).clip(lower=1)
    df['wickets_remaining'] = 10 - df['c_team_wickets']

    # Run rate uses balls actually bowled
    df['current_run_rate'] = runs * 6 / balls.where(balls > 0)        # NaN on first ball

    df['projected_score'] = runs + df['current_run_rate'] * df['balls_remaining'] / 6
    df['balls_per_wicket_remaining'] = df['balls_remaining'] / df['wickets_remaining']
    return df


df = add_match_state_features(df)

In [62]:
bad_inn = short[['match_id', 'innings']].drop_duplicates().assign(short_inn=1)
tmp = df.merge(bad_inn, on=['match_id', 'innings'], how='left')
ok = tmp['short_inn'].isna()

# 2. Legal-ball count agrees with over number, outside short-over innings    Expected: 0
print('2.', ((tmp['c_team_balls'] // 6 != tmp['over']) & ok).sum())

# 2b. New balls_remaining equals the old formula outside short-over innings   Expected: 0
print('2b.', ((tmp['balls_remaining'] != 120 - tmp['c_team_balls']) & ok).sum())

# 3. ball_in_over agrees with legal-ball count, outside short-over innings    Expected: 0
print('3.', ((tmp['ball_in_over'] != tmp['c_team_balls'] % 6 + 1) & ok).sum())

#drop a helped column
df = df.drop(columns=['_is_wkt'])

2. 0
2b. 0
3. 0


## SUBGROUP 2

In [63]:
def add_chase_features(df, cfg=CONFIG):
    """Subgroup 2. Chase-only features: NaN in innings 1. Uses only state entering the ball."""
    df = df.copy()
    chasing = df['innings'] == 2

    df['runs_needed'] = (df['runs_target'] - df['c_team_runs']).where(chasing)
    df['required_run_rate'] = df['runs_needed'] * 6 / df['balls_remaining']   # uncapped
    df['pressure_diff'] = df['required_run_rate'] - df['current_run_rate']    # NaN on chase's first ball
    df['runs_needed_per_wicket'] = df['runs_needed'] / df['wickets_remaining']
    return df


df = add_chase_features(df)

In [64]:
new = ['runs_needed', 'required_run_rate', 'pressure_diff', 'runs_needed_per_wicket']
i1, i2 = df[df['innings'] == 1], df[df['innings'] == 2]
g2 = i2.groupby('match_id')

# 1. Everything is NaN in innings 1                       Expected: 0
print('1.', i1[new + ['runs_target']].notna().sum().sum())

# 2. runs_target is present and constant in every chase   Expected: 0 True
print('2.', i2['runs_target'].isna().sum(), (g2['runs_target'].nunique() == 1).all())

# 3. NaNs in innings 2 only where intended                Expected: 0 0 0 | [0]
print('3.', i2[['runs_needed', 'required_run_rate', 'runs_needed_per_wicket']].isna().sum().tolist(),
      '|', i2.loc[i2['pressure_diff'].isna(), 'c_team_balls'].unique())

# 4. The chase never continues after the target is reached Expected: True
print('4.', (i2['runs_needed'] >= 1).all())

# 5. First delivery of each chase                          Expected: True True
f = g2.head(1)
print('5.', (f['runs_needed'] == f['runs_target']).all(),
      np.allclose(f['required_run_rate'], f['runs_target'] * 6 / 120))

# 6. Runs needed never goes up within a chase              Expected: 0
print('6.', (g2['runs_needed'].diff() > 0).sum())

# 7. Ranges and distribution
print('7.', (i2['runs_needed_per_wicket'] > 0).all())     # Expected: True
print(i2['required_run_rate'].describe())
print('RRR > 18 rows:', (i2['required_run_rate'] > 18).sum(),
      '| matches:', i2.loc[i2['required_run_rate'] > 18, 'match_id'].nunique())

# 8. Dtypes                                                Expected: all float64
print('8.'); print(df[new].dtypes)

# 9. Hand-check: last 2 overs of one chase
m = i2['match_id'].iloc[0]
cols = ['over', 'ball', 'valid_ball', 'runs_target', 'c_team_runs', 'c_team_wickets',
        'balls_remaining', 'current_run_rate', 'runs_needed', 'required_run_rate',
        'pressure_diff', 'runs_needed_per_wicket']
chase = i2[i2['match_id'] == m]
print('9.'); print(chase[chase['over'] >= chase['over'].max() - 1][cols])

1. 0
2. 0 True
3. [0, 0, 0] | [0]
4. False
5. True True
6. 0
7. False
count    142300.000000
mean         10.818113
std          12.955320
min          -3.600000
25%           7.333333
50%           9.100000
75%          11.294118
max         792.000000
Name: required_run_rate, dtype: float64
RRR > 18 rows: 8392 | matches: 535
8.
runs_needed               float64
required_run_rate         float64
pressure_diff             float64
runs_needed_per_wicket    float64
dtype: object
9.
     over  ball  valid_ball  runs_target  c_team_runs  c_team_wickets  \
216    14     1           0        223.0           77               9   
217    14     1           1        223.0           78               9   
218    14     2           1        223.0           78               9   
219    14     3           1        223.0           78               9   
220    14     4           1        223.0           79               9   
221    14     5           1        223.0           80               9   
222 

In [65]:
df['target'] = df['target'].astype(str)

##SUBGROUP 3

In [66]:
def add_momentum_features(df, cfg=CONFIG):
    """Subgroup 3. Team momentum. Uses only deliveries before the current one, within the same innings.
    Relies on rows being sorted by match_id, innings, over, ball (stable order for wide + re-bowl)."""
    df = df.copy()
    keys = [df['match_id'], df['innings']]
    pos = df.groupby(['match_id', 'innings'], sort=False).cumcount()   # deliveries already bowled this innings

    # One value per delivery for each quantity we want to count
    per_ball = pd.DataFrame({
        'runs':   df['ball_runs_total'],
        'wkts':   df['is_dismissal'],
        'dots':   ((df['valid_ball'] == 1) & (df['ball_runs_total'] == 0) & (df['is_dismissal'] == 0)).astype(int),
        'bdry':   df['ball_runs_batter'].isin([4, 6]).astype(int),
        'extras': df['ball_runs_extras'],
        'legal':  df['valid_ball'],
    }, index=df.index).astype(float)

    # Totals before this delivery, within the innings
    prev = per_ball.groupby(keys).cumsum() - per_ball

    def window(k):
        """Sum over the previous k deliveries; NaN until k deliveries have been bowled."""
        out = prev - prev.groupby(keys).shift(k)
        out.loc[pos < k, :] = np.nan
        return out

    w6, w12 = window(6), window(12)
    for name in ['runs', 'wkts', 'dots', 'bdry']:
        df[f'team_{name}_last6'] = w6[name]
        df[f'team_{name}_last12'] = w12[name]
    df['team_extras_last12'] = w12['extras']

    def deliveries_since(event):
        """Deliveries bowled since the last event in this innings (0 = previous delivery);
        if no event yet, deliveries bowled so far."""
        last = pos.where(event == 1).groupby(keys).shift(1)
        last = last.groupby(keys).ffill()
        return (pos - last - 1).fillna(pos).astype(int)

    df['balls_since_last_boundary'] = deliveries_since(per_ball['bdry'])
    df['balls_since_last_wicket'] = deliveries_since(per_ball['wkts'])

    # Chase only: required rate minus the true run rate over the last 12 deliveries
    recent_rr = w12['runs'] * 6 / w12['legal'].where(w12['legal'] > 0)
    df['recent_pressure_diff'] = (df['required_run_rate'] - recent_rr).where(df['innings'] == 2)

    # Runs already scored in the current over (optional, ablation)
    over_keys = ['match_id', 'innings', 'over']
    df['runs_this_over'] = df.groupby(over_keys)['ball_runs_total'].cumsum() - df['ball_runs_total']

    return df


df = add_momentum_features(df)

In [67]:
keys = [df['match_id'], df['innings']]
pos = df.groupby(['match_id', 'innings'], sort=False).cumcount()
w6 = [f'team_{n}_last6' for n in ['runs', 'wkts', 'dots', 'bdry']]
w12 = [f'team_{n}_last12' for n in ['runs', 'wkts', 'dots', 'bdry']] + ['team_extras_last12']

# 0. Rows are in the expected order                                  Expected: True
order = df.sort_values(['match_id', 'innings', 'over', 'ball'], kind='stable').index
print('0.', (order == df.index).all())

# 1. Windows are NaN exactly until they are full                     Expected: True True
print('1.', all((df[c].isna() == (pos < 6)).all() for c in w6),
            all((df[c].isna() == (pos < 12)).all() for c in w12))

# 2. Window values within physical limits                            Expected: True True
print('2.', all(df[c].dropna().between(0, 6).all() for c in ['team_wkts_last6', 'team_dots_last6', 'team_bdry_last6']),
            all(df[c].dropna().between(0, 12).all() for c in ['team_wkts_last12', 'team_dots_last12', 'team_bdry_last12']))

# 3. Independent cross-check on 100 random innings (slow rolling method)   Expected: True True True
rng = np.random.default_rng(0)
inn_ids = df[['match_id', 'innings']].drop_duplicates().sample(100, random_state=0)
sub = df.merge(inn_ids, on=['match_id', 'innings'])
roll = lambda col, k: sub.groupby(['match_id', 'innings'])[col].transform(
    lambda s: s.shift(1).rolling(k, min_periods=k).sum())
sub_dots = ((sub['valid_ball'] == 1) & (sub['ball_runs_total'] == 0) & (sub['is_dismissal'] == 0)).astype(int)
sub['_dot'] = sub_dots
print('3.', np.allclose(roll('ball_runs_total', 6), sub['team_runs_last6'], equal_nan=True),
            np.allclose(roll('is_dismissal', 12), sub['team_wkts_last12'], equal_nan=True),
            np.allclose(roll('_dot', 12), sub['team_dots_last12'], equal_nan=True))

# 4. balls_since_* follow their rule on every row                    Expected: True True True
def since_ok(col, event):
    prev_evt = event.groupby(keys).shift(1)
    prev_val = df[col].groupby(keys).shift(1)
    expected = np.where(prev_evt == 1, 0, prev_val + 1)
    m = (pos > 0).values
    return (df[col].values[m] == expected[m]).all()
print('4.', since_ok('balls_since_last_boundary', df['ball_runs_batter'].isin([4, 6]).astype(int)),
            since_ok('balls_since_last_wicket', df['is_dismissal']),
            (df.loc[pos == 0, ['balls_since_last_boundary', 'balls_since_last_wicket']] == 0).all().all())

# 5. runs_this_over: 0 at the start of each over, then adds the previous delivery   Expected: True True
ok = ['match_id', 'innings', 'over']
prev_rto = df.groupby(ok)['runs_this_over'].shift(1)
prev_runs = df.groupby(ok)['ball_runs_total'].shift(1)
first = prev_rto.isna()
print('5.', (df.loc[first, 'runs_this_over'] == 0).all(),
            (df.loc[~first, 'runs_this_over'] == prev_rto[~first] + prev_runs[~first]).all())

# 6. recent_pressure_diff: empty in innings 1; in innings 2 NaN only for the first 12 deliveries   Expected: 0 True
i2 = df['innings'] == 2
print('6.', df.loc[~i2, 'recent_pressure_diff'].notna().sum(),
            (df.loc[i2, 'recent_pressure_diff'].isna() == (pos[i2] < 12)).all())

# 7. Shape and types                                                  Expected: 64 columns, no object dtype in new columns
new = w6 + w12 + ['balls_since_last_boundary', 'balls_since_last_wicket', 'recent_pressure_diff', 'runs_this_over']
print('7.', df.shape[1], len(new)); print(df[new].dtypes.value_counts())



0. True
1. True True
2. True True
3. True True True
4. True True True
5. True True
6. 0 True
7. 64 13
float64    10
int64       3
Name: count, dtype: int64


##SUBGROUP 4

In [68]:
def add_striker_features(df, cfg=CONFIG):
    """Subgroup 4. Striker's state in this innings, using only balls before the current one.
    Relies on rows being sorted by match_id, innings, over, ball."""
    df = df.copy()
    gid = df.groupby(['match_id', 'innings', 'batter'], sort=False).ngroup()   # one id per batter-innings
    faced = df['binary_ball_faced'] == 1
    dot = ((df['valid_ball'] == 1) & (df['ball_runs_batter'] == 0)).astype(float)   # batter's dot

    # Strike rate before this ball; NaN before the first ball faced
    df['batter_sr_innings'] = df['c_batter_runs'] * 100 / df['c_batter_balls'].where(df['c_batter_balls'] > 0)

    # Work on faced balls only: values as they stand AFTER each faced ball
    fg = gid[faced]
    n_faced = fg.groupby(fg).cumcount() + 1                     # faced balls so far, including this one

    def after_window(x, k=6):
        cs = x.groupby(fg).cumsum()
        out = cs - cs.groupby(fg).shift(k, fill_value=0)
        return out.where(n_faced >= k)                          # NaN until k balls faced

    runs6 = after_window(df.loc[faced, 'ball_runs_batter'].astype(float))
    dots6 = after_window(dot[faced])
    breaks = (dot[faced] == 0).astype(int).groupby(fg).cumsum()  # any faced non-dot ends the streak
    streak = dot[faced].groupby([fg, breaks]).cumsum()

    def from_last_faced(after_vals):
        """For every row, the value after the striker's most recent faced ball before this row."""
        s = pd.Series(np.nan, index=df.index)
        s.loc[faced] = after_vals
        s = s.groupby(gid).shift(1)
        return s.groupby(gid).ffill()

    df['batter_runs_last6'] = from_last_faced(runs6)
    df['batter_dots_last6'] = from_last_faced(dots6)
    df['batter_dots_in_row'] = from_last_faced(streak).fillna(0).astype(int)

    # Share of the batter's runs that came from boundaries; NaN while they have 0 runs
    bdry_runs = df['ball_runs_batter'].where(df['ball_runs_batter'].isin([4, 6]), 0)
    prev_bdry = bdry_runs.groupby(gid).cumsum() - bdry_runs
    df['batter_bdry_run_pct'] = prev_bdry / df['c_batter_runs'].where(df['c_batter_runs'] > 0)

    return df


df = add_striker_features(df)

In [69]:
gid = df.groupby(['match_id', 'innings', 'batter'], sort=False).ngroup()
faced = df['binary_ball_faced'] == 1
new = ['batter_sr_innings', 'batter_runs_last6', 'batter_dots_last6', 'batter_dots_in_row', 'batter_bdry_run_pct']

# 1. The c_ columns agree with the per-ball data (the new features rely on them)   Expected: True True
prev_runs = df['ball_runs_batter'].groupby(gid).cumsum() - df['ball_runs_batter']
prev_faced = df['binary_ball_faced'].groupby(gid).cumsum() - df['binary_ball_faced']
print('1.', (prev_runs == df['c_batter_runs']).all(), (prev_faced == df['c_batter_balls']).all())

# 2. NaN exactly where intended                                       Expected: True True True True
print('2.', (df['batter_runs_last6'].isna() == (df['c_batter_balls'] < 6)).all(),
            (df['batter_dots_last6'].isna() == (df['c_batter_balls'] < 6)).all(),
            (df['batter_sr_innings'].isna() == (df['c_batter_balls'] == 0)).all(),
            (df['batter_bdry_run_pct'].isna() == (df['c_batter_runs'] == 0)).all())

# 3. Ranges                                                           Expected: True True True True
print('3.', df['batter_runs_last6'].dropna().between(0, 36).all(),
            df['batter_dots_last6'].dropna().between(0, 6).all(),
            (df['batter_dots_in_row'].between(0, 10**6) & (df['batter_dots_in_row'] <= df['c_batter_balls'])).all(),
            df['batter_bdry_run_pct'].dropna().between(0, 1).all())

# 4. Independent cross-check on 100 random innings (plain Python loop)   Expected: True True True
def slow(sub):
    hist, out = {}, []
    for r in sub.itertuples():
        h = hist.setdefault(r.batter, [])
        full = len(h) >= 6
        runs6 = sum(x[0] for x in h[-6:]) if full else np.nan
        dots6 = sum(x[1] for x in h[-6:]) if full else np.nan
        streak = 0
        for x in reversed(h):
            if x[1] == 1: streak += 1
            else: break
        out.append((runs6, dots6, streak))
        if r.binary_ball_faced == 1:
            h.append((r.ball_runs_batter, int(r.valid_ball == 1 and r.ball_runs_batter == 0)))
    return pd.DataFrame(out, index=sub.index, columns=['r6', 'd6', 'st'])

inn_ids = df[['match_id', 'innings']].drop_duplicates().sample(100, random_state=0)
sub = df.merge(inn_ids.assign(_k=1), on=['match_id', 'innings']).set_index(
      df.index[df.set_index(['match_id', 'innings']).index.isin(inn_ids.set_index(['match_id', 'innings']).index)])
ref = pd.concat([slow(g) for _, g in sub.groupby(['match_id', 'innings'], sort=False)])
print('4.', np.allclose(ref['r6'], sub.loc[ref.index, 'batter_runs_last6'], equal_nan=True),
            np.allclose(ref['d6'], sub.loc[ref.index, 'batter_dots_last6'], equal_nan=True),
            (ref['st'] == sub.loc[ref.index, 'batter_dots_in_row']).all())

# 5. A wide doesn't change the striker's features                     Expected: True
nxt_faced = df.groupby(gid)['binary_ball_faced'].shift(-1).eq(1)
m = (~faced) & nxt_faced
nxt = df.groupby(gid)[new[1:4]].shift(-1)
print('5.', all(np.allclose(df.loc[m, c], nxt.loc[m, c], equal_nan=True) for c in new[1:4]))

# 6. Shape and types                                                  Expected: 69 columns, all numeric
print('6.', df.shape[1]); print(df[new].dtypes)


1. True True
2. True True True True
3. True True True True
4. True True True
5. True
6. 69
batter_sr_innings      float64
batter_runs_last6      float64
batter_dots_last6      float64
batter_dots_in_row       int64
batter_bdry_run_pct    float64
dtype: object


##SUBGROUP 5

In [70]:
def add_partnership_features(df, cfg=CONFIG):
    """Subgroup 5. Partnership and non-striker state, using only balls before the current one.
    Relies on rows being sorted by match_id, innings, over, ball."""
    df = df.copy()
    keys = [df['match_id'], df['innings']]

    # Partnership segments: a new one starts on the delivery after any batter leaves.
    # player_out (not is_dismissal) on purpose: a retirement also ends the partnership.
    left = df['player_out'].notna().astype(int)
    seg = left.groupby(keys).shift(1, fill_value=0).groupby(keys).cumsum()
    pkeys = [df['match_id'], df['innings'], seg]

    df['partnership_runs'] = df['ball_runs_total'].groupby(pkeys).cumsum() - df['ball_runs_total']   # includes extras
    df['partnership_balls'] = df['valid_ball'].groupby(pkeys).cumsum() - df['valid_ball']             # legal balls

    # Non-striker's runs and balls: their totals after the last delivery they faced as striker
    row = pd.Series(np.arange(len(df)), index=df.index, name='_row')
    as_striker = pd.DataFrame({
        'match_id': df['match_id'], 'innings': df['innings'], 'non_striker': df['batter'], '_row': row,
        'ns_runs': df['c_batter_runs'] + df['ball_runs_batter'],
        'ns_balls': df['c_batter_balls'] + df['binary_ball_faced'],
    })
    lookup = pd.DataFrame({'match_id': df['match_id'], 'innings': df['innings'],
                           'non_striker': df['non_striker'], '_row': row})
    found = pd.merge_asof(lookup, as_striker, on='_row', by=['match_id', 'innings', 'non_striker'],
                          allow_exact_matches=False, direction='backward')

    df['non_striker_runs'] = found['ns_runs'].fillna(0).astype(int).values    # 0 if they haven't faced yet
    df['non_striker_balls'] = found['ns_balls'].fillna(0).astype(int).values
    df['non_striker_sr'] = df['non_striker_runs'] * 100 / df['non_striker_balls'].where(df['non_striker_balls'] > 0)

    return df


df = add_partnership_features(df)

In [71]:
# Left-right pair: 1 if the striker and non-striker bat with different hands.
# Hands come from the dataframe's batter_hand (the corrected version), not player_metadata.csv.
print('max hands per batter:', df.groupby('batter')['batter_hand'].nunique().max())   # expect 1
hand = df.groupby('batter')['batter_hand'].first()

ns_hand = df['non_striker'].map(hand)
print('non-strikers with no hand (never faced a ball):', df.loc[ns_hand.isna(), 'non_striker'].unique(),
      '|', ns_hand.isna().sum(), 'rows')
ns_hand = ns_hand.fillna('RHB')   # a handful of rows; RHB is the majority hand

df['is_left_right_pair'] = (df['batter_hand'] != ns_hand).astype(int)
print(df['is_left_right_pair'].value_counts(normalize=True))

max hands per batter: 1
non-strikers with no hand (never faced a ball): ['A Nel' 'C Ganapathy' 'TP Sudhindra' 'BW Hilfenhaus' 'Mayank Rawat'] | 8 rows
is_left_right_pair
1    0.50687
0    0.49313
Name: proportion, dtype: float64


In [72]:
keys = [df['match_id'], df['innings']]
new = ['partnership_runs', 'partnership_balls', 'non_striker_runs', 'non_striker_balls', 'non_striker_sr']
first = df.groupby(['match_id', 'innings']).cumcount() == 0
prev = df.groupby(['match_id', 'innings'])[['player_out', 'ball_runs_total', 'valid_ball', 'partnership_runs',
                                             'partnership_balls', 'batter', 'non_striker',
                                             'non_striker_runs', 'non_striker_balls']].shift(1)
after_exit = prev['player_out'].notna()

# 1. Partnerships start at 0: innings start and the delivery after a batter leaves   Expected: True True
print('1.', (df.loc[first, ['partnership_runs', 'partnership_balls']] == 0).all().all(),
            (df.loc[after_exit, ['partnership_runs', 'partnership_balls']] == 0).all().all())

# 2. Otherwise they add the previous delivery                       Expected: True True
cont = ~first & ~after_exit
print('2.', (df.loc[cont, 'partnership_runs'] == prev.loc[cont, 'partnership_runs'] + prev.loc[cont, 'ball_runs_total']).all(),
            (df.loc[cont, 'partnership_balls'] == prev.loc[cont, 'partnership_balls'] + prev.loc[cont, 'valid_ball']).all())

# 3. A partnership can't exceed the team's totals                   Expected: True True
print('3.', (df['partnership_runs'] <= df['c_team_runs']).all(), (df['partnership_balls'] <= df['c_team_balls']).all())

# 4. When strike rotates, the new striker's c_ columns equal what we showed as the non-striker   Expected: True True
swap = ~first & (df['batter'] == prev['non_striker'])
print('4.', (df.loc[swap, 'c_batter_runs'] == prev.loc[swap, 'non_striker_runs']).all(),
            (df.loc[swap, 'c_batter_balls'] == prev.loc[swap, 'non_striker_balls']).all())

# 5. Independent cross-check on 100 random innings (plain Python loop)   Expected: True True
def slow(sub):
    tot, out = {}, []
    for r in sub.itertuples():
        out.append(tot.get(r.non_striker, (0, 0)))
        rs, bs = tot.get(r.batter, (0, 0))
        tot[r.batter] = (rs + r.ball_runs_batter, bs + r.binary_ball_faced)
    return pd.DataFrame(out, index=sub.index, columns=['r', 'b'])

inn_ids = set(map(tuple, df[['match_id', 'innings']].drop_duplicates().sample(100, random_state=0).values))
mask = [k in inn_ids for k in zip(df['match_id'], df['innings'])]
ref = pd.concat([slow(g) for _, g in df[mask].groupby(['match_id', 'innings'], sort=False)])
print('5.', (ref['r'] == df.loc[ref.index, 'non_striker_runs']).all(),
            (ref['b'] == df.loc[ref.index, 'non_striker_balls']).all())

# 6. NaN only where intended                                        Expected: 0 True
print('6.', df[new[:4]].isna().sum().sum(),
            (df['non_striker_sr'].isna() == (df['non_striker_balls'] == 0)).all())

# 7. Shape and types                                                Expected: 74 columns, all numeric
print('7.', df.shape[1]); print(df[new].dtypes)

# 9. Left-right flag is 0/1 and complete                           Expected: True
print('9.', df['is_left_right_pair'].isin([0, 1]).all())

1. True True
2. True True
3. True True
4. True True
5. True True
6. 0 True
7. 75
partnership_runs       int64
partnership_balls      int64
non_striker_runs       int64
non_striker_balls      int64
non_striker_sr       float64
dtype: object
9. True


In [73]:
df.iloc[-250:-220]

,match_id,innings,batting_team,bowling_team,over,ball,batter,bat_pos,ball_runs_batter,binary_ball_faced,...,batter_runs_last6,batter_dots_last6,batter_dots_in_row,batter_bdry_run_pct,partnership_runs,partnership_balls,non_striker_runs,non_striker_balls,non_striker_sr,is_left_right_pair
295303,1535464,2,Gujarat Titans,Rajasthan Royals,16,1,JC Buttler,3,1,1,...,6.0,0.0,0,0.000000,7,6,5,4,125.0,1
295304,1535464,2,Gujarat Titans,Rajasthan Royals,16,2,Washington Sundar,4,0,1,...,NaN,NaN,0,0.800000,8,7,9,8,112.5,1
295305,1535464,2,Gujarat Titans,Rajasthan Royals,16,3,Washington Sundar,4,4,1,...,NaN,NaN,1,0.800000,8,8,9,8,112.5,1
295306,1535464,2,Gujarat Titans,Rajasthan Royals,16,4,Washington Sundar,4,6,1,...,9.0,3.0,0,0.888889,12,9,9,8,112.5,1
295307,1535464,2,Gujarat Titans,Rajasthan Royals,16,5,Washington Sundar,4,1,1,...,15.0,2.0,0,0.933333,18,10,9,8,112.5,1
295308,1535464,2,Gujarat Titans,Rajasthan Royals,16,6,JC Buttler,3,0,1,...,6.0,0.0,0,0.000000,19,11,16,8,200.0,1
295309,1535464,2,Gujarat Titans,Rajasthan Royals,17,1,Washington Sundar,4,0,1,...,15.0,2.0,0,0.875000,19,12,9,9,100.0,1
295310,1535464,2,Gujarat Titans,Rajasthan Royals,17,2,R Tewatia,5,0,1,...,NaN,NaN,0,NaN,0,0,9,9,100.0,1
295311,1535464,2,Gujarat Titans,Rajasthan Royals,17,3,R Tewatia,5,0,1,...,NaN,NaN,1,NaN,0,1,9,9,100.0,1
295312,1535464,2,Gujarat Titans,Rajasthan Royals,17,4,R Tewatia,5,0,1,...,NaN,NaN,2,NaN,0,2,9,9,100.0,1


In [74]:
df.to_parquet(DATA + 'ipl_features.parquet', index=False)